# Manufacturing Multi-Agent Quality Assistant — Synthetic Data Generation

Generates 5 tables of synthetic manufacturing data for the multi-agent demo:
1. **quality_inspections** — 500 inspection records with messy inspector notes
2. **service_bulletins** — 100 technical service bulletins (text-heavy, for Vector Search)
3. **defect_history** — 2,000 historical defect records (for SQL agent)
4. **corrective_actions** — 300 action records linked to inspections
5. **plant_metrics** — Daily KPI rollups (FTQ, OEE, throughput) for 6 months

All data is synthetic — no real customer data.


## Setup
Loads catalog and schema from job parameters (or notebook widgets). Creates the schema in an existing catalog you can write to. It does not create a catalog.


In [ ]:
%run ./_config


Re-read job widgets in this notebook so catalog and schema are defined after `%run`.


In [ ]:
# Job parameters / widgets must be read in this notebook. %run does not always export names from .ipynb.
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
try:
    llm_endpoint = dbutils.widgets.get("llm_endpoint").strip()
except Exception:
    llm_endpoint = "databricks-claude-sonnet-4-5"
try:
    embedding_endpoint = dbutils.widgets.get("embedding_endpoint").strip()
except Exception:
    embedding_endpoint = "databricks-gte-large-en"
try:
    vs_endpoint = dbutils.widgets.get("vs_endpoint").strip()
except Exception:
    vs_endpoint = "mfg_quality_vs_endpoint"
try:
    eval_sample_size = dbutils.widgets.get("eval_sample_size").strip() or "50"
except Exception:
    eval_sample_size = "50"
try:
    cost_owner = dbutils.widgets.get("cost_owner").strip()
except Exception:
    cost_owner = ""
VS_INDEX_NAME = f"{catalog}.{schema}.service_bulletins_index"
print(f"Bound widgets: {catalog}.{schema}")


Creates the demo schema if it is missing, then sets it as the current schema.


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")
print(f"Using: {catalog}.{schema}")


## Libraries and seed
Standard Python libraries for synthetic data. Seed is fixed so reruns are stable.


In [ ]:
import random
import json
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import col

random.seed(42)


## Reference Data


In [ ]:
PLANTS = [
    {"plant_id": "P001", "plant_name": "Arlington Assembly", "city": "Arlington", "state": "TX"},
    {"plant_id": "P002", "plant_name": "Flint Assembly", "city": "Flint", "state": "MI"},
    {"plant_id": "P003", "plant_name": "Fort Wayne Assembly", "city": "Fort Wayne", "state": "IN"},
    {"plant_id": "P004", "plant_name": "Wentzville Assembly", "city": "Wentzville", "state": "MO"},
    {"plant_id": "P005", "plant_name": "Spring Hill Manufacturing", "city": "Spring Hill", "state": "TN"},
]

PART_TYPES = [
    {"part_type": "Brake Rotor", "part_family": "Braking", "spec_unit": "mm", "spec_min": 28.0, "spec_max": 32.0, "critical_threshold": 25.0},
    {"part_type": "Door Panel", "part_family": "Body", "spec_unit": "mm", "spec_min": 1.0, "spec_max": 1.5, "critical_threshold": 0.7},
    {"part_type": "Wiring Harness", "part_family": "Electrical", "spec_unit": "ohms", "spec_min": 0.1, "spec_max": 0.5, "critical_threshold": 1.0},
    {"part_type": "Engine Block", "part_family": "Powertrain", "spec_unit": "mm", "spec_min": 95.0, "spec_max": 95.5, "critical_threshold": 94.0},
    {"part_type": "Paint Surface", "part_family": "Body", "spec_unit": "microns", "spec_min": 80.0, "spec_max": 120.0, "critical_threshold": 60.0},
    {"part_type": "Transmission Gear", "part_family": "Powertrain", "spec_unit": "mm", "spec_min": 45.0, "spec_max": 45.3, "critical_threshold": 44.5},
    {"part_type": "Exhaust Manifold", "part_family": "Exhaust", "spec_unit": "mm", "spec_min": 3.0, "spec_max": 5.0, "critical_threshold": 2.0},
    {"part_type": "Battery Module", "part_family": "Electrical", "spec_unit": "V", "spec_min": 3.6, "spec_max": 4.2, "critical_threshold": 3.0},
]

INSPECTORS = [
    "M. Johnson", "R. Patel", "T. Williams", "S. Chen", "K. Martinez",
    "D. Brown", "A. Garcia", "J. Lee", "L. Thompson", "P. Anderson",
    "C. Davis", "N. Wilson", "E. Taylor", "F. Robinson", "H. Clark"
]

SHIFTS = ["Day", "Swing", "Night"]
LINES = ["Line 1", "Line 2", "Line 3", "Line 4", "Line 5"]

ROOT_CAUSES = ["Material", "Process", "Equipment", "Human"]
SEVERITIES = ["Critical", "Major", "Minor", "None"]
SEVERITY_WEIGHTS = [0.08, 0.20, 0.32, 0.40]  # realistic distribution


## Table 1: quality_inspections (500 records)

Messy, realistic inspector notes with ground truth labels for eval.


In [ ]:
def generate_measurement(part, severity):
    """Generate a measurement value based on severity."""
    spec_min = float(part["spec_min"])
    spec_max = float(part["spec_max"])
    spec_mid = (spec_min + spec_max) / 2
    spec_range = spec_max - spec_min

    if severity == "None":
        return round(random.uniform(spec_min, spec_max), 2)
    elif severity == "Minor":
        if random.random() < 0.5:
            return round(spec_min - random.uniform(0, spec_range * 0.3), 2)
        else:
            return round(spec_max + random.uniform(0, spec_range * 0.3), 2)
    elif severity == "Major":
        if random.random() < 0.5:
            return round(spec_min - random.uniform(spec_range * 0.3, spec_range * 1.0), 2)
        else:
            return round(spec_max + random.uniform(spec_range * 0.3, spec_range * 1.0), 2)
    else:  # Critical
        if random.random() < 0.5:
            return round(spec_min - random.uniform(spec_range * 1.0, spec_range * 2.0), 2)
        else:
            return round(spec_max + random.uniform(spec_range * 1.0, spec_range * 2.0), 2)


def generate_root_cause(severity):
    """Assign root cause with realistic distribution per severity."""
    if severity == "None":
        return "None"
    elif severity == "Critical":
        return random.choices(ROOT_CAUSES, weights=[15, 25, 40, 20])[0]
    elif severity == "Major":
        return random.choices(ROOT_CAUSES, weights=[25, 30, 25, 20])[0]
    else:  # Minor
        return random.choices(ROOT_CAUSES, weights=[20, 20, 20, 40])[0]


def generate_inspector_note(part, severity, root_cause, measurement, plant, line):
    """Generate messy, realistic inspector notes."""
    unit = part["spec_unit"]
    spec_min = float(part["spec_min"])
    spec_max = float(part["spec_max"])
    part_name = part["part_type"]

    # Templates by severity — messy, abbreviated, realistic
    if severity == "None":
        templates = [
            f"PASS - {part_name} measured {measurement}{unit}. Within spec [{spec_min}-{spec_max}]. No issues.",
            f"{part_name} OK. {measurement}{unit}, spec range {spec_min}-{spec_max}. Approved for assembly.",
            f"Inspected {part_name} on {line}. Measurement: {measurement}{unit}. PASS. Good condition.",
            f"ACCEPT - {part_name} @ {measurement}{unit}. Meets tolerance. Released to next station.",
            f"{part_name} checked. {measurement}{unit} is within bounds. All good, moving on.",
            f"Visual + dimensional check {part_name}. {measurement}{unit}. No defects observed. PASS.",
        ]
    elif severity == "Minor":
        cause_notes = {
            "Material": f"slight surface blemish on {part_name}, possibly supplier material batch issue",
            "Process": f"minor deviation in {part_name} - feed rate may need adjustment",
            "Equipment": f"{part_name} borderline - caliper may need recalibration, reading {measurement}{unit}",
            "Human": f"operator missed initial check on {part_name}, caught on re-inspect at {measurement}{unit}",
        }
        templates = [
            f"BORDERLINE - {part_name} at {measurement}{unit} ({unit}), spec is [{spec_min}-{spec_max}]. {cause_notes[root_cause]}. Log and monitor.",
            f"{part_name} measured {measurement}{unit}. Slightly outside spec [{spec_min}-{spec_max}]. {cause_notes[root_cause]}. Flagged for review.",
            f"WARN - {part_name} {measurement}{unit}. Marginal. {cause_notes[root_cause]}. Logging but not pulling from line.",
            f"Minor issue {part_name}. Measured {measurement}{unit}, expected {spec_min}-{spec_max}. {cause_notes[root_cause]}. Cosmetic/borderline - monitor trend.",
        ]
    elif severity == "Major":
        cause_notes = {
            "Material": f"raw material defect suspected - {part_name} shows inconsistent composition at {measurement}{unit}",
            "Process": f"process parameter drift on {line} - {part_name} at {measurement}{unit}, need to check temp/pressure settings",
            "Equipment": f"tooling wear detected - {part_name} consistently measuring {measurement}{unit}, tool replacement needed on {line}",
            "Human": f"wrong setup loaded for {part_name} batch on {line} - measuring {measurement}{unit} instead of [{spec_min}-{spec_max}]",
        }
        templates = [
            f"REJECT - {part_name} measured {measurement}{unit} ({unit}), WAY outside spec [{spec_min}-{spec_max}]. {cause_notes[root_cause]}. Rework required before shipment.",
            f"FAIL - {part_name} at {measurement}{unit}. Spec [{spec_min}-{spec_max}]. {cause_notes[root_cause]}. Pulling batch for rework.",
            f"{part_name} FAILED inspection. {measurement}{unit} vs required [{spec_min}-{spec_max}]. {cause_notes[root_cause]}. DO NOT ship. Rework queue.",
            f"OUT OF SPEC - {part_name} {measurement}{unit}. {cause_notes[root_cause]}. Entire lot from {line} needs rework. Notifying supervisor.",
        ]
    else:  # Critical
        cause_notes = {
            "Material": f"CRITICAL material failure - {part_name} structural integrity compromised at {measurement}{unit}. Suspect contaminated supplier batch",
            "Process": f"CRITICAL process failure on {line} - {part_name} at {measurement}{unit}. Complete process breakdown, safety risk",
            "Equipment": f"CRITICAL equipment failure - {part_name} at {measurement}{unit}. Machine malfunction on {line}, possible safety hazard. SHUT DOWN LINE",
            "Human": f"CRITICAL operator error - {part_name} at {measurement}{unit}. Wrong tooling installed on {line}. Safety risk, immediate stop",
        }
        templates = [
            f"REJECT - {part_name} measured {measurement}{unit} ({unit}), WAY outside spec [{spec_min}-{spec_max}]. {cause_notes[root_cause]}. Entire batch from {line} suspect. Shut down line for investigation.",
            f"SAFETY ALERT - {part_name} at {measurement}{unit}. CRITICAL. Spec [{spec_min}-{spec_max}]. {cause_notes[root_cause]}. Quarantine all units from this batch. Notify plant manager.",
            f"CRITICAL DEFECT - {part_name} {measurement}{unit}. {cause_notes[root_cause]}. LINE SHUT DOWN. All units from past 2 hours quarantined. Escalating to quality manager.",
            f"STOP - {part_name} structural failure detected. {measurement}{unit} vs [{spec_min}-{spec_max}]. {cause_notes[root_cause]}. BATCH QUARANTINE. Do not release.",
        ]

    return random.choice(templates)


# Generate 500 inspections
inspections = []
base_date = datetime(2025, 9, 1)

for i in range(500):
    severity = random.choices(SEVERITIES, weights=SEVERITY_WEIGHTS)[0]
    part = random.choice(PART_TYPES)
    plant = random.choice(PLANTS)
    line = random.choice(LINES)
    root_cause = generate_root_cause(severity)
    measurement = generate_measurement(part, severity)
    inspector = random.choice(INSPECTORS)
    shift = random.choice(SHIFTS)
    inspection_date = base_date + timedelta(days=random.randint(0, 180))
    inspection_time = f"{random.randint(6,22):02d}:{random.randint(0,59):02d}"

    note = generate_inspector_note(part, severity, root_cause, measurement, plant, line)

    inspections.append({
        "inspection_id": f"INS-{i+1:05d}",
        "inspection_date": inspection_date.strftime("%Y-%m-%d"),
        "inspection_time": inspection_time,
        "plant_id": plant["plant_id"],
        "plant_name": plant["plant_name"],
        "line": line,
        "shift": shift,
        "inspector": inspector,
        "part_type": part["part_type"],
        "part_family": part["part_family"],
        "measurement_value": float(measurement),
        "measurement_unit": part["spec_unit"],
        "spec_min": float(part["spec_min"]),
        "spec_max": float(part["spec_max"]),
        "defect_notes": note,
        # Ground truth — AI never sees these
        "expected_severity": severity,
        "expected_root_cause": root_cause,
    })

df_inspections = spark.createDataFrame(inspections)
df_inspections.write.mode("overwrite").saveAsTable("quality_inspections")
print(f"✅ quality_inspections: {df_inspections.count()} rows")
display(df_inspections.limit(5))


## Table 2: service_bulletins (100 records)

Multi-paragraph technical documents for Vector Search retrieval.


In [ ]:
def generate_service_bulletin(idx, part, plant):
    """Generate a realistic, multi-paragraph service bulletin."""
    bulletin_id = f"SB-{datetime.now().year}-{idx+1:04d}"
    failure_modes = {
        "Brake Rotor": ["excessive wear", "warping under heat cycling", "thickness variation", "surface cracking", "corrosion pitting"],
        "Door Panel": ["gap misalignment", "weld fracture at hinge point", "paint adhesion failure", "trim clip breakage", "water intrusion"],
        "Wiring Harness": ["intermittent continuity failure", "insulation degradation", "connector pin corrosion", "chafing against body panel", "wrong pin-out configuration"],
        "Engine Block": ["bore diameter out of spec", "coolant passage blockage", "casting porosity", "main bearing cap misalignment", "oil gallery restriction"],
        "Paint Surface": ["orange peel texture", "fish-eye contamination", "color mismatch between panels", "clear coat delamination", "dirt inclusion"],
        "Transmission Gear": ["tooth profile deviation", "hardness variation", "spline wear pattern", "bearing race scoring", "synchronizer ring damage"],
        "Exhaust Manifold": ["exhaust leak at gasket surface", "thermal cracking", "catalyst substrate damage", "heat shield detachment", "weld porosity"],
        "Battery Module": ["cell voltage imbalance", "thermal runaway risk", "BMS communication fault", "swelling detected", "electrolyte leakage"],
    }

    failure = random.choice(failure_modes[part["part_type"]])
    affected_models = random.sample(["Silverado", "Sierra", "Tahoe", "Suburban", "Equinox", "Blazer", "Colorado", "Canyon", "Hummer EV", "Bolt EUV"], k=random.randint(1, 3))
    model_years = f"20{random.randint(23,26)}-20{random.randint(26,27)}"

    root_cause_detail = random.choice([
        f"Investigation revealed that supplier batch {random.randint(1000,9999)} from {random.choice(['Tier 1 supplier A', 'Tier 2 supplier B', 'overseas casting facility'])} contained material outside specification limits.",
        f"Root cause traced to {random.choice(['calibration drift', 'worn tooling', 'incorrect parameter upload', 'software update regression'])} on {random.choice(LINES)} during {random.choice(['third shift', 'weekend overtime run', 'model changeover'])}.",
        f"Failure mode linked to {random.choice(['design tolerance stack-up', 'environmental exposure during shipping', 'assembly sequence error', 'insufficient torque specification'])}. Engineering change notice pending.",
        f"Analysis indicates {random.choice(['fatigue failure under cyclic loading', 'galvanic corrosion from dissimilar metals', 'thermal expansion mismatch', 'residual stress from heat treatment'])}.",
    ])

    fix_procedure = random.choice([
        f"1. Remove and discard affected {part['part_type']}.\n2. Install replacement part from updated supplier batch (P/N suffix -R2 or later).\n3. Verify measurement within spec [{part['spec_min']}-{part['spec_max']}{part['spec_unit']}].\n4. Document rework in MES system.",
        f"1. Inspect all units from affected production dates.\n2. Measure critical dimension — accept if within [{part['spec_min']}-{part['spec_max']}{part['spec_unit']}].\n3. For out-of-spec units: route to rework station for {random.choice(['re-machining', 're-coating', 'component replacement', 'harness re-termination'])}.\n4. Update build record with rework code RW-{random.randint(100,999)}.",
        f"1. Quarantine all affected inventory.\n2. Perform 100% inspection using {random.choice(['CMM', 'optical comparator', 'go/no-go gauge', 'megger test'])}.\n3. Sort: pass units return to line, fail units scrap.\n4. Implement containment at receiving inspection for next 3 supplier shipments.",
    ])

    severity_level = random.choice(["Safety", "Compliance", "Quality", "Quality"])
    priority = random.choice(["Immediate", "Urgent", "Standard"])

    text = f"""SERVICE BULLETIN {bulletin_id}

SUBJECT: {part['part_type']} — {failure}
AFFECTED MODELS: {', '.join(affected_models)} ({model_years})
AFFECTED PLANTS: {plant['plant_name']}
SEVERITY: {severity_level} | PRIORITY: {priority}
DATE ISSUED: {(datetime.now() - timedelta(days=random.randint(1, 365))).strftime('%Y-%m-%d')}

PROBLEM DESCRIPTION:
{part['part_type']} units have been identified with {failure}. Measurements show values outside the acceptable range of {part['spec_min']}-{part['spec_max']}{part['spec_unit']}. This condition may result in {random.choice(['reduced component life', 'assembly interference', 'customer-noticeable defect', 'potential safety concern', 'warranty claim exposure'])}.

ROOT CAUSE:
{root_cause_detail}

CORRECTIVE ACTION:
{fix_procedure}

PARTS AFFECTED: Estimated {random.randint(50, 2000)} units in field + {random.randint(100, 5000)} in plant inventory.
ENGINEERING CONTACT: {random.choice(['J. Smith', 'M. Gonzalez', 'R. Kim', 'S. Patel'])} (ext. {random.randint(2000, 9999)})
"""
    return {
        "bulletin_id": bulletin_id,
        "part_type": part["part_type"],
        "part_family": part["part_family"],
        "failure_mode": failure,
        "plant_id": plant["plant_id"],
        "plant_name": plant["plant_name"],
        "severity_level": severity_level,
        "priority": priority,
        "affected_models": ", ".join(affected_models),
        "model_years": model_years,
        "bulletin_text": text,
        "issue_date": (datetime.now() - timedelta(days=random.randint(1, 365))).strftime("%Y-%m-%d"),
    }


bulletins = []
for i in range(100):
    part = random.choice(PART_TYPES)
    plant = random.choice(PLANTS)
    bulletins.append(generate_service_bulletin(i, part, plant))

df_bulletins = spark.createDataFrame(bulletins)
df_bulletins.write.mode("overwrite").saveAsTable("service_bulletins")
print(f"✅ service_bulletins: {df_bulletins.count()} rows")
display(df_bulletins.limit(3))


## Table 3: defect_history (2,000 records)

Historical defect records for the SQL agent to query trends, patterns, and aggregations.


In [ ]:
defect_statuses = ["Open", "In Rework", "Reworked", "Scrapped", "Released"]
defect_status_weights = [0.05, 0.10, 0.50, 0.15, 0.20]

defects = []
base_date = datetime(2025, 3, 1)

for i in range(2000):
    part = random.choice(PART_TYPES)
    plant = random.choice(PLANTS)
    severity = random.choices(SEVERITIES[:3], weights=[0.10, 0.30, 0.60])[0]  # no None in defect history
    root_cause = random.choices(ROOT_CAUSES, weights=[25, 30, 25, 20])[0]
    defect_date = base_date + timedelta(days=random.randint(0, 365))
    resolution_days = random.randint(0, 14) if severity != "Critical" else random.randint(0, 3)
    status = random.choices(defect_statuses, weights=defect_status_weights)[0]
    cost = round(random.uniform(50, 500) if severity == "Minor" else random.uniform(500, 5000) if severity == "Major" else random.uniform(5000, 50000), 2)
    batch_size = random.randint(20, 500)
    affected_count = random.randint(1, min(batch_size, 50))

    defects.append({
        "defect_id": f"DEF-{i+1:06d}",
        "defect_date": defect_date.strftime("%Y-%m-%d"),
        "plant_id": plant["plant_id"],
        "plant_name": plant["plant_name"],
        "line": random.choice(LINES),
        "shift": random.choice(SHIFTS),
        "part_type": part["part_type"],
        "part_family": part["part_family"],
        "severity": severity,
        "root_cause": root_cause,
        "status": status,
        "batch_id": f"BATCH-{random.randint(10000, 99999)}",
        "batch_size": batch_size,
        "affected_units": affected_count,
        "rework_cost_usd": cost,
        "resolution_days": resolution_days,
        "inspector": random.choice(INSPECTORS),
    })

df_defects = spark.createDataFrame(defects)
df_defects.write.mode("overwrite").saveAsTable("defect_history")
print(f"✅ defect_history: {df_defects.count()} rows")
display(df_defects.groupBy("severity").count().orderBy("severity"))


## Table 4: corrective_actions (300 records)

Actions taken in response to inspections — linked to inspection IDs.


In [ ]:
action_types = [
    "Rework", "Scrap", "Quarantine", "Line Shutdown", "Tool Replacement",
    "Process Adjustment", "Supplier Notification", "Retrain Operator",
    "Recalibrate Equipment", "Engineering Review"
]

action_statuses = ["Planned", "In Progress", "Completed", "Verified"]

# Link to inspections that had defects (non-None severity)
defect_inspection_ids = [insp["inspection_id"] for insp in inspections if insp["expected_severity"] != "None"]

actions = []
for i in range(300):
    insp_id = random.choice(defect_inspection_ids)
    insp = next(x for x in inspections if x["inspection_id"] == insp_id)
    action_date = datetime.strptime(insp["inspection_date"], "%Y-%m-%d") + timedelta(hours=random.randint(1, 72))
    completion_date = action_date + timedelta(days=random.randint(0, 7)) if random.random() > 0.2 else None

    actions.append({
        "action_id": f"ACT-{i+1:05d}",
        "inspection_id": insp_id,
        "plant_id": insp["plant_id"],
        "part_type": insp["part_type"],
        "action_type": random.choice(action_types),
        "action_description": f"Corrective action for {insp['part_type']} defect on {insp['line']} at {insp['plant_name']}. Root cause: {insp['expected_root_cause']}.",
        "assigned_to": random.choice(INSPECTORS),
        "action_date": action_date.strftime("%Y-%m-%d"),
        "completion_date": completion_date.strftime("%Y-%m-%d") if completion_date else None,
        "status": random.choice(action_statuses),
        "severity": insp["expected_severity"],
    })

df_actions = spark.createDataFrame(actions)
df_actions.write.mode("overwrite").saveAsTable("corrective_actions")
print(f"✅ corrective_actions: {df_actions.count()} rows")
display(df_actions.groupBy("action_type").count().orderBy(col("count").desc()).limit(10))


## Table 5: plant_metrics (900 records — 5 plants x 180 days)

Daily KPI rollups for each plant — FTQ rate, OEE, throughput, downtime.


In [ ]:
metrics = []
base_date = datetime(2025, 9, 1)

for plant in PLANTS:
    # Each plant has a baseline performance profile
    base_ftq = random.uniform(0.92, 0.98)
    base_oee = random.uniform(0.75, 0.90)
    base_throughput = random.randint(400, 800)

    for day in range(180):
        date = base_date + timedelta(days=day)
        is_weekend = date.weekday() >= 5

        # Add realistic variation
        daily_ftq = min(1.0, max(0.80, base_ftq + random.gauss(0, 0.015)))
        daily_oee = min(1.0, max(0.50, base_oee + random.gauss(0, 0.03)))
        daily_throughput = max(0, int(base_throughput * (0.3 if is_weekend else 1.0) + random.gauss(0, 30)))
        planned_downtime = random.randint(0, 60) if not is_weekend else random.randint(0, 480)
        unplanned_downtime = max(0, int(random.expovariate(0.05))) if random.random() > 0.7 else 0
        scrap_count = max(0, int(daily_throughput * (1 - daily_ftq)))
        rework_count = max(0, int(scrap_count * random.uniform(0.3, 0.7)))

        # Occasionally simulate a bad day (equipment failure, etc.)
        if random.random() < 0.03:
            daily_ftq *= 0.85
            daily_oee *= 0.70
            unplanned_downtime += random.randint(60, 240)

        metrics.append({
            "metric_date": date.strftime("%Y-%m-%d"),
            "plant_id": plant["plant_id"],
            "plant_name": plant["plant_name"],
            "ftq_rate": round(daily_ftq, 4),
            "oee_rate": round(daily_oee, 4),
            "throughput_units": daily_throughput,
            "scrap_count": scrap_count,
            "rework_count": rework_count,
            "planned_downtime_min": planned_downtime,
            "unplanned_downtime_min": unplanned_downtime,
            "total_downtime_min": planned_downtime + unplanned_downtime,
            "shift_count": 1 if is_weekend else 3,
        })

df_metrics = spark.createDataFrame(metrics)
df_metrics.write.mode("overwrite").saveAsTable("plant_metrics")
print(f"✅ plant_metrics: {df_metrics.count()} rows")
display(df_metrics.filter(col("plant_id") == "P001").orderBy("metric_date").limit(10))


## Summary


In [ ]:
tables = ["quality_inspections", "service_bulletins", "defect_history", "corrective_actions", "plant_metrics"]
for t in tables:
    count = spark.table(t).count()
    print(f"  {catalog}.{schema}.{t}: {count:,} rows")

print(f"\n✅ All 5 tables created in {catalog}.{schema}")
print(f"\nNext steps:")
print(f"  1. Run 02_Vector_Search_Index.py — Create vector index on service_bulletins")
print(f"  2. Run 03_Sub_Agents.py — Build classifier, retriever, and SQL sub-agents")
print(f"  3. Run 04_Supervisor_Agent.py — Build supervisor with routing logic")
print(f"  4. Optional: Run 05_Eval_And_Cost.py — Evaluate accuracy and track costs")
print(f"  5. Open the Databricks App `mfg-quality-assistant`")
